<a href="https://colab.research.google.com/github/eliponc-code/sim-mat-2025/blob/main/PA1_UV_Vis_EO_O2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###*Espectroscopías Ópticas*

##Práctica de Análisis 1 (PA1): Análisis de espectros UV-Vis mediante los métodos de Swanepoel y Tauc


#### **Objetivo**
Interpretar espectros UV-Visible de películas delgadas y utilizarlos para determinar
propiedades ópticas, como el índice de refracción, el espesor y el ancho de banda
prohibida óptico, mediante los métodos de Swanepoel y Tauc-plot





**Ejercicio 1.** Realiza una investigación en distintas fuentes sobre el uso de la espectroscopía UV-Visible en el análisis de materiales y explica, con tus propias palabras, sus principales fundamentos y aplicaciones. Puedes orientar la investigación hacia el análisis de películas delgadas o de materiales nanoestructurados en polvo, de
acuerdo con el tema de mayor interés.


Describe algunas de las propiedades que pueden estudiarse mediante esta técnica y
menciona aplicaciones relevantes de los materiales analizados mediante
espectroscopía UV-Visible.


Incluye las referencias bibliográficas utilizadas al final del reporte

**Fundamento:** La espectroscopía UV-Vis mide cuánta luz absorbe, transmite o refleja una muestra en función de la longitud de onda. El espectro da información sobre las transiciones electrónicas del material. Para una película delgada sobre un sustrato transparente se usa el modo de transmisión, donde la transmitancia es T = I/I₀. Los polvos nanoestructurados o las muestras opacas dispersan mucha luz, por lo que requieren reflectancia difusa. La absorbancia y el coeficiente de absorción de una película de espesor *d* se calculan como: (Chen y Jaramillo, 2017)

$$A = -\log_{10}(T)$$

$$\alpha = \frac{2.303 \cdot A}{d}$$

Como los espectros se registran en nm, la energía del fotón se obtiene con hν (eV) = 1239.8 / λ (nm)

**Propiedades que se pueden estudiar:** En una película delgada, las franjas de interferencia del espectro de transmitancia dependen del índice de refracción *n* y del espesor *d*. El método de Swanepoel usa los máximos y mínimos de esas franjas para estimar *n* y *d* en la región de débil absorción. Con *d* conocido se obtiene *α* y se estima el ancho de banda prohibida óptico ($E_g$) con el método de Tauc, que es la práctica habitual en semiconductores (Makuła et al., 2018). Cerca del borde de absorción se cumple:

$$(\alpha h\nu)^{1/n} \propto h\nu - E_g$$

Aquí $n = 1/2$ corresponde a una transición directa permitida y n = 2 a una indirecta permitida. La técnica tiene limitaciones: la reflexión y la dispersión elevan la absorción aparente. Además, ajustar la recta en la cola exponencial del borde ($α < ~10⁴ cm⁻¹$) subestima $E_g$, por lo que conviene elegir bien la región lineal.

**Materiales y aplicaciones:** El $Ta_2 O_5$ en película delgada se ha caracterizado por transmitancia: en películas de sol-gel se obtuvieron $n ≈ 1.70 – 1.72$ a 550 nm y $E_g$ ≈ 3.75 eV, con aplicación como conductor protónico en dispositivos electrocrómicos (Tepehan et al., 1999). El $TiO_2$ es un semiconductor tipo *n* con alto índice de refracción, usado como recubrimiento óptico y guía de onda, además de fotoánodo en celdas solares y en fotocatálisis (Stephen, 2020). Sus propiedades dependen de la fase: el $TiO_2$ amorfo tiene $E_g ≈ 3.5 eV$ y $n ≈ 1.9–2 a 600 nm$, mientras que el anatasa tiene $E_g ≈ 3.2 eV$ y un índice mayor (Stephen, 2020). Por eso ambas películas deben interpretarse en términos de fase, espesor y estructura electrónica.

**Ejercicio 2.** Se dispone del espectro de transmitancia de una película delgada de
pentóxido de tántalo ($Ta_2 O_5$) depositada sobre un sustrato de cuarzo. Utiliza los
archivos proporcionados correspondientes a:

- Película Ta2O5 sobre cuarzo: Datos-1.txt
- Sustrato de cuarzo: Datos-S.txt
Aplica el método de Swanepoel para determinar el índice de refracción y el espesor de la película.

Para el tratamiento de los datos, utiliza la transmitancia en escala de 0 a 1 y
considera las condiciones de aplicación del método estudiadas durante el curso.

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.stats import linregress
import matplotlib.pyplot as plt

long = np.genfromtxt('Datos-S.txt', usecols=0, skip_header=1, delimiter='\t')
Transmitancia_s = np.genfromtxt('Datos-S.txt', usecols=1, skip_header=1, delimiter='\t')
Transmitancia_film = np.genfromtxt('Datos-1.txt', usecols=1, skip_header=1, delimiter='\t')

l=long
Ts, Tf = Transmitancia_s*1, Transmitancia_film*1

#transmitancia de la muestra y el sustrato sin recubrir
fig = go.Figure()
fig.add_trace(go.Scatter(x = l, y = Tf, mode='lines', name='Ta2O5/Cuarzo'))
fig.add_trace(go.Scatter(x = l, y = Ts, mode='lines', name='Sustrato'))
fig.update_yaxes(range=[0,1], title_text='Transmitancia')
fig.update_xaxes(range=[200, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=700, height=500)
fig.show()

In [3]:
#máximos y minimos
inicio=np.where(l==300)[0][0]
final=np.where(l==1000)[0][0]+1

T_film=Transmitancia_film[inicio:final]
L=l[inicio:final]

maximos, _ = find_peaks(T_film, height=0)
minimos, _ = find_peaks(-T_film, height=-np.inf)

print('Máximos en:')
for i in maximos:
    print(L[i], 'nm')

print('Mínimos en:')
for i in minimos:
    print(L[i], 'nm')

Máximos en:
444.0 nm
534.0 nm
702.0 nm
Mínimos en:
471.0 nm
597.0 nm
833.0 nm


In [4]:
sup_env_func = interp1d(L[maximos], T_film[maximos], kind='linear',  bounds_error=False, fill_value='extrapolate')
inf_env_func = interp1d(L[minimos], T_film[minimos], kind='linear',  bounds_error=False, fill_value='extrapolate')

sup_env = sup_env_func(L)
inf_env = inf_env_func(L)

In [5]:
#me marcaba error en sqrt pq habia negativos, tmb puse linear
sup_real = np.maximum(sup_env, inf_env)
inf_real = np.minimum(sup_env, inf_env)

sup_env = sup_real
inf_env = inf_real

print('R env sup:',
      np.min(sup_env), 'a', np.max(sup_env))

print('R env inf:',
      np.min(inf_env), 'a', np.max(inf_env))

print('puntos con sup_env < inf_env:',
      np.sum(sup_env < inf_env))

R env sup: 0.4375876482857142 a 1.076240954107143
R env inf: 0.31562173399999993 a 0.7737946149915255
puntos con sup_env < inf_env: 0


In [6]:
Ts=0.966 #constante del ejemplo

s=1/Ts+((1/Ts**2)-1)**(1/2)

N = 2 * s * ((sup_env - inf_env)/(sup_env * inf_env)) + ((s ** 2 + 1) / 2)
print("s =", s)
print("N mínimo =", np.min(N))
print("N máximo =", np.max(N))
print("N² - s² mínimo =", np.min(N**2 - s**2))

s = 1.3028385778568707
N mínimo = 1.3532893245142512
N máximo = 3.649749845606777
N² - s² mínimo = 0.13400363589212505


In [7]:
n = (N + (N ** 2 - s ** 2) ** (1/2)) ** (1/2)

In [8]:
#índice de refracción
fig = go.Figure()
fig.add_trace(go.Scatter(x = L, y = n, mode='lines', name='n de Ta2O5'))
fig.update_yaxes(range=[0,3], title_text='Índice de refracción')
fig.update_xaxes(range=[300, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=900, height=500)
fig.show()

In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = l, y = Tf, mode='lines', name='Ta2O5/Sustrato'))
fig.add_trace(go.Scatter(x = L, y = sup_env, mode='lines', name='Envolvente superior'))
fig.add_trace(go.Scatter(x = L, y = inf_env, mode='lines', name='Envolvente inferior'))
fig.update_yaxes(range=[0,1], title_text='Transmitancia')
fig.update_xaxes(range=[200, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=900, height=500)
fig.show()

In [10]:
#Espesor usando el segundo y tercer máximo en la transmitancia
l1_max = L[maximos[1]]
l2_max = L[maximos[2]]
n1_max = n[maximos[1]]
n2_max = n[maximos[2]]

d_max = (l1_max*l2_max) / (2 * (l2_max*n1_max - l1_max*n2_max))

print('Espesor de:')
print(round(d_max, 2), 'nm')

Espesor de:
501.93 nm


In [11]:
#Verificación usando el segundo y tercer mínimo de la transmitancia
l1_min = L[minimos[1]]
l2_min = L[minimos[2]]
n1_min = n[minimos[1]]
n2_min = n[minimos[2]]

d_min = (l1_min*l2_min) / (2 * (l2_min*n1_min - l1_min*n2_min))

print('Espesor de:')
print(round(d_min, 2), 'nm')

Espesor de:
626.07 nm


**Ejercicio 3.** El archivo Datos-2.txt contiene el espectro de transmitancia de una película
delgada de dióxido de titanio ($TiO_2$) depositada sobre un sustrato de cuarzo. Aplica el
método de Swanepoel para determinar el índice de refracción y el espesor de la
película. Utiliza para el sustrato las mismas consideraciones empleadas en el ejercicio
anterior.

In [12]:
Transmitancia_film_Ti = np.genfromtxt('Datos-2.txt', usecols=1, delimiter = '\t', skip_header=1)
Tf_Ti = Transmitancia_film_Ti*1 #Los datos ya están en fracción (0 a 1)

In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = l, y = Tf_Ti, mode='lines', name='TiO2/Sustrato'))
fig.add_trace(go.Scatter(x = l, y = Transmitancia_s, mode='lines', name='Sustrato'))
fig.update_yaxes(range=[0,1], title_text='Transmitancia')
fig.update_xaxes(range=[200, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=700, height=500)
fig.show()

In [14]:
Min = np.where(l==380)[0][0]
Max = np.where(l==1000)[0][0]
T_film = Tf_Ti[Min:Max]
L = l[Min:Max]

In [15]:
maximos, _ = find_peaks(T_film, height=0)
minimos, _ = find_peaks(-T_film, height=-np.inf)
print('Máximos en:')
for i in maximos:
    print(L[i], 'nm')

print('Mínimos en:')
for i in minimos:
    print(L[i], 'nm')

Máximos en:
381.0 nm
450.0 nm
572.0 nm
834.0 nm
Mínimos en:
412.0 nm
500.0 nm
678.0 nm


In [16]:
sup_env_func = interp1d(L[maximos], T_film[maximos], kind='quadratic', fill_value='extrapolate')
inf_env_func = interp1d(L[minimos], T_film[minimos], kind='quadratic', fill_value='extrapolate')

sup_env = sup_env_func(L)
inf_env = inf_env_func(L)

In [17]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = l, y = Tf_Ti, mode='lines', name='TiO2/Sustrato'))
fig.add_trace(go.Scatter(x = L, y = sup_env, mode='lines', name='Envolvente superior'))
fig.add_trace(go.Scatter(x = L, y = inf_env, mode='lines', name='Envolvente inferior'))
fig.update_yaxes(range=[0,1], title_text='Transmitancia')
fig.update_xaxes(range=[200, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=900, height=500)
fig.show()

In [18]:
Ts = np.mean(Transmitancia_s[Min:Max]) #Se toma constante
TM_env = sup_env/1
Tm_env = inf_env/1
print('Ts =', round(Ts, 4))

Ts = 0.9661


In [19]:
s = 1 / Ts + ((1 / Ts ** 2) - 1) ** (1/2)
N = 2 * s * ((TM_env - Tm_env)/(TM_env * Tm_env)) + ((s ** 2 + 1) / 2)
n = (N + (N ** 2 - s ** 2) ** (1/2)) ** (1/2)

In [20]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = L, y = n, mode='lines', name='n de TiO2'))
fig.update_yaxes(range=[0,3], title_text='Índice de refracción')
fig.update_xaxes(range=[300, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=900, height=500)
fig.show()

In [21]:
print('n en los máximos:')
for i in maximos:
    print(L[i], 'nm  n =', round(n[i], 3))

print('n en los mínimos:')
for i in minimos:
    print(L[i], 'nm  n =', round(n[i], 3))

n en los máximos:
381.0 nm  n = 2.178
450.0 nm  n = 2.089
572.0 nm  n = 1.991
834.0 nm  n = 2.067
n en los mínimos:
412.0 nm  n = 2.135
500.0 nm  n = 2.04
678.0 nm  n = 1.974


In [22]:
l1_max = L[maximos[1]]
l2_max = L[maximos[2]]
n1_max = n[maximos[1]]
n2_max = n[maximos[2]]
d_max_Ti = (l1_max*l2_max) / (2 * (l2_max*n1_max - l1_max*n2_max))
print('Espesor de:')
print(round(d_max_Ti, 2), 'nm')

Espesor de:
430.15 nm


In [23]:
l1_min = L[minimos[1]]
l2_min = L[minimos[2]]
n1_min = n[minimos[1]]
n2_min = n[minimos[2]]
d_min_Ti = (l1_min*l2_min) / (2 * (l2_min*n1_min - l1_min*n2_min))
print('Espesor de:')
print(round(d_min_Ti, 2), 'nm')

Espesor de:
427.74 nm


**Ejercicio 4.** Utiliza los espesores determinados para las películas de $Ta_2 O_5$ y $TiO_2$ en los ejercicios anteriores y sus respectivos espectros de transmitancia para estimar el ancho
de banda prohibida óptico, ($E_g$), mediante el método de Tauc-plot. Realiza el análisis considerando:
- Transiciones directas permitidas.
- Transiciones indirectas permitidas.


A partir de los datos experimentales, realiza el tratamiento necesario para obtener la información requerida por el método. Presenta las gráficas de Tauc correspondientes y los valores de $E_g$ obtenidos para cada material y tipo de transición.

In [24]:
def intervalo(x, xmin, xmax):
      idx = np.where((x > xmin) & (x < xmax))[0]
      return idx

In [25]:
T = Tf
A = - np.log(T)

In [26]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = l, y = A, mode='lines', name='Película Ta2O5'))
fig.update_yaxes(range=[0,35], title_text='Absorbancia')
fig.update_xaxes(range=[200, 1000], title_text='Longitud de onda (nm)')
fig.update_layout(autosize=False, width=900, height=500)
fig.show()

In [27]:
#coeficiente de absorción y hv, con d_max Ta2O5
d_cm = d_max * 1e-7 #cm
alph = 2.303 * A / d_cm
hv = 1239.8 / l

In [28]:
ahv_2 = (alph * hv) ** 2 #directa
ahv_1_2 = (alph * hv) ** (1/2) #indirecta

In [29]:
fig = make_subplots(rows=1, cols=2,subplot_titles=["Transición directa", "Transición indirecta"])
fig.add_trace(go.Scatter(x=hv, y=ahv_2, mode='lines', name=r'$(\alpha h\nu)^2$'),row=1, col=1)
fig.update_xaxes(title_text='hv (eV)', range=[1,7], row=1, col=1)
fig.update_yaxes(title_text=r'$(\alpha h \nu)^2$', row=1, col=1)

# s2 transición indirecta
fig.add_trace(go.Scatter(x=hv, y=ahv_1_2, mode='lines', name=r'$(\alpha h\nu)^{1/2}$'),row=1, col=2)
fig.update_xaxes(title_text='hv (eV)', range=[1,7], row=1, col=2)
fig.update_yaxes(title_text=r'$(\alpha h \nu)^{1/2}$', row=1, col=2)

fig.update_layout(autosize=False,width=900, height=400, showlegend=False)
fig.show()

In [30]:
mask_dir = intervalo(hv, 5.3, 6.0) # t directa
mask_ind = intervalo(hv, 4.6, 5.4) # t indirecta

x_dir = hv[mask_dir]
y_dir = ahv_2[mask_dir] # (ahv)^2 --> t directa

x_ind = hv[mask_ind]
y_ind = ahv_1_2[mask_ind] # (ahv)^(1/2) --> t indirecta

In [32]:
slope_dir, intercept_dir, r_dir, *_ = linregress(x_dir, y_dir)
slope_ind, intercept_ind, r_ind, *_ = linregress(x_ind, y_ind)

if abs(slope_dir) < 1e-12:
    raise RuntimeError("Pendiente (slope_dir) demasiado cercana a cero; revisar la selección de la región lineal.")
Eg_dir = -intercept_dir / slope_dir

if abs(slope_ind) < 1e-12:
    raise RuntimeError("Pendiente (slope_ind) demasiado cercana a cero; revisar la selección de la región lineal.")
Eg_ind = -intercept_ind / slope_ind

print(f"Eg (directo): {Eg_dir:.2f} eV  (R2 = {r_dir**2:.4f})")
print(f"Eg (indirecto): {Eg_ind:.2f} eV  (R2 = {r_ind**2:.4f})")

Eg (directo): 5.01 eV  (R2 = 0.9985)
Eg (indirecto): 3.94 eV  (R2 = 0.9990)


In [33]:
x_min_dir = min(min(x_dir), Eg_dir) - 0.3
x_max_dir = max(max(x_dir), Eg_dir) + 0.3
x_fit_dir = np.linspace(x_min_dir, x_max_dir, 200)
y_fit_dir = slope_dir * x_fit_dir + intercept_dir

# Para el caso indirecto
x_min_ind = min(min(x_ind), Eg_ind) - 0.3
x_max_ind = max(max(x_ind), Eg_ind) + 0.3
x_fit_ind = np.linspace(x_min_ind, x_max_ind, 200)
y_fit_ind = slope_ind * x_fit_ind + intercept_ind

In [34]:
fig = make_subplots(rows=1, cols=2,subplot_titles=["Transición directa", "Transición indirecta"])

fig.add_trace(go.Scatter(x=hv, y=ahv_2, mode='lines', name='Datos directos', line=dict(color='blue')),row=1, col=1)
fig.add_trace(go.Scatter(x=x_fit_dir, y=y_fit_dir, mode='lines', name='Ajuste lineal directo', line=dict(color='red', dash='dash')),row=1, col=1)
fig.add_trace(go.Scatter(x=[Eg_dir, Eg_dir], y=[0, max(y_fit_dir)], mode='lines', line=dict(color='gray', dash='dot'), name=f'Eg directo = {Eg_dir:.2f} eV'), row=1, col=1)
fig.update_xaxes(title_text='hv (eV)', range=[1, 7], row=1, col=1)
fig.update_yaxes(title_text=r'$(\alpha h\nu)^2$', row=1, col=1)

fig.add_trace(go.Scatter(x=hv, y=ahv_1_2, mode='lines', name='Datos indirectos', line=dict(color='blue')),row=1, col=2)
fig.add_trace(go.Scatter(x=x_fit_ind, y=y_fit_ind, mode='lines', name='Ajuste lineal indirecto', line=dict(color='red', dash='dash')),row=1, col=2)
fig.add_trace(go.Scatter(x=[Eg_ind, Eg_ind], y=[0, max(y_fit_ind)], mode='lines', line=dict(color='gray', dash='dot'), name=f'Eg indirecto = {Eg_ind:.2f} eV'),row=1, col=2)
fig.update_xaxes(title_text='hv (eV)', range=[1, 7], row=1, col=2)
fig.update_yaxes(title_text=r'$(\alpha h\nu)^{1/2}$', row=1, col=2)

fig.update_layout(width=1000, height=420, showlegend=True)
fig.show()

In [35]:
Eg_dir_Ta = Eg_dir
Eg_ind_Ta = Eg_ind

**Ejercicio 5.** A partir de los resultados obtenidos para las películas de $Ta_2 O_5$ y $TiO_2$,
realiza una comparación de sus propiedades ópticas. Analiza las diferencias
observadas en sus espectros de transmitancia, índice de refracción, espesor y ancho
de banda prohibida. Relaciona los resultados con las características ópticas y
electrónicas de cada material. Finalmente, determina cuál de los dos materiales
presenta características más favorables para una aplicación óptica específica y sustenta
la elección a partir de los resultados obtenidos

In [36]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=l, y=Tf, mode="lines", name="Ta2O5/Sustrato", line=dict(width=2)))
fig.add_trace(go.Scatter(x=l, y=Tf_Ti, mode="lines", name="TiO2/Sustrato", line=dict(width=2)))
fig.add_trace(go.Scatter(x=l, y=Transmitancia_s, mode="lines", name="Sustrato", line=dict(width=2)))
fig.update_layout(title=dict(text="Comparación de espectros de transmitancia", font=dict(size=18),x=0.5,
    ),xaxis=dict(title=dict(text="Longitud de onda (nm)", font=dict(size=14)),showgrid=True,gridcolor="rgba(0,0,0,0.1)",  # Équivalente a alpha=0.3
    ),yaxis=dict(title=dict(text="Transmitancia", font=dict(size=14)),showgrid=True,gridcolor="rgba(0,0,0,0.1)",
    ),width=800, height=500,template="plotly_white", legend=dict(font=dict(size=12)),)
fig.show()


#####**Referencias:**
1. Chen, Z., y Jaramillo, T. F. (2017). The use of UV-visible spectroscopy to measure the band gap of a semiconductor. Stanford University. https://mmrc.caltech.edu/Cary%20UV-Vis%20Int.Sphere/Literature/Spectroscopy%20Jaramillo.pdf

2. Davis, E. A., y Mott, N. F. (1970). Conduction in non-crystalline systems V. Conductivity, optical absorption and photoconductivity in amorphous semiconductors. Philosophical Magazine, 22, 903.

3. Makuła, P., Pacia, M., y Macyk, W. (2018). How to correctly determine the band gap energy of modified semiconductor photocatalysts based on UV–Vis spectra. The Journal of Physical Chemistry Letters, 9(23), 6814–6817. https://doi.org/10.1021/acs.jpclett.8b02892

4. Stephen, L. (2020). Titanium dioxide versatile solid crystalline: An overview. En R. S. Dongre y D. R. Peshwe (Eds.), Assorted dimensional reconfigurable materials. IntechOpen. https://doi.org/10.5772/intechopen.92056

